# Mortgage Default Analysis

This notebook predicts mortgage default using Logistic Regression, Random Forest, and XGBoost. It removes leakage columns, handles class imbalance, and compares model performance using accuracy, ROC-AUC, precision, recall, and F1-score.

**Execution note:** The full dataset is loaded for inspection. For faster execution in normal systems/Colab, the modelling section uses a stratified sample of 80,000 records while preserving the default/non-default ratio.

## 1. Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)

pd.set_option("display.max_columns", None)

## 2. Load Dataset

In [2]:
DATA_PATH = "mortgage.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (622489, 23)


## 3. Target Distribution

In [3]:
target_col = "default_time"
print("Target distribution:")
print(df[target_col].value_counts())
print("\nDefault rate:", round(df[target_col].mean() * 100, 2), "%")

Target distribution:
0    607331
1     15158
Name: default_time, dtype: int64

Default rate: 2.44 %


## 4. Leakage Correction

The target is `default_time`. The following columns are removed from the feature set:

- `id`: identifier, not a business predictor
- `default_time`: target variable
- `payoff_time`: future/outcome information
- `status_time`: future/outcome information
- `orig_time`: removed to keep the modelling setup conservative and avoid time-index leakage

This prevents the model from learning information that would not be available at prediction time.

In [4]:
leakage_cols = ["id", "default_time", "payoff_time", "status_time", "orig_time"]
X_full = df.drop(columns=leakage_cols)
y_full = df[target_col].astype(int)

print("Features used:")
print(X_full.columns.tolist())
print("\nNumber of features:", X_full.shape[1])

Features used:
['time', 'first_time', 'mat_time', 'balance_time', 'LTV_time', 'interest_rate_time', 'hpi_time', 'gdp_time', 'uer_time', 'REtype_CO_orig_time', 'REtype_PU_orig_time', 'REtype_SF_orig_time', 'investor_orig_time', 'balance_orig_time', 'FICO_orig_time', 'LTV_orig_time', 'Interest_Rate_orig_time', 'hpi_orig_time']

Number of features: 18


## 5. Stratified Sample for Faster Modelling

In [5]:
SAMPLE_SIZE = 80000

if len(df) > SAMPLE_SIZE:
    sample_df, _ = train_test_split(
        df, train_size=SAMPLE_SIZE, stratify=df[target_col], random_state=42
    )
else:
    sample_df = df.copy()

X = sample_df.drop(columns=leakage_cols)
y = sample_df[target_col].astype(int)

print("Modelling sample shape:", sample_df.shape)
print("Sample default rate:", round(y.mean() * 100, 2), "%")

Modelling sample shape: (80000, 23)
Sample default rate: 2.44 %


## 6. Train-Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training default rate:", round(y_train.mean() * 100, 2), "%")
print("Testing default rate:", round(y_test.mean() * 100, 2), "%")

Training shape: (64000, 18)
Testing shape: (16000, 18)
Training default rate: 2.43 %
Testing default rate: 2.44 %


## 7. Preprocessing and Class Weighting

In [7]:
numeric_features = X.columns.tolist()

scaled_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)
])

tree_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features)
])

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

print("Negative class count:", negative_count)
print("Positive class count:", positive_count)
print("XGBoost scale_pos_weight:", round(scale_pos_weight, 2))

Negative class count: 62442
Positive class count: 1558
XGBoost scale_pos_weight: 40.08


## 8. Build Models

In [8]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", scaled_preprocess),
        ("model", LogisticRegression(max_iter=100, class_weight="balanced", solver="lbfgs", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", RandomForestClassifier(
            n_estimators=20, max_depth=8, min_samples_leaf=50,
            class_weight="balanced_subsample", n_jobs=2, random_state=42
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", XGBClassifier(
            n_estimators=40, max_depth=3, learning_rate=0.10,
            subsample=0.80, colsample_bytree=0.80,
            eval_metric="logloss", tree_method="hist",
            scale_pos_weight=scale_pos_weight, n_jobs=2, random_state=42
        ))
    ])
}

## 9. Train and Evaluate Models

In [9]:
results = []
fitted_models = {}

for model_name, model in models.items():
    print("\n" + "="*70)
    print(model_name)
    print("="*70)
    
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Precision_Default": precision_score(y_test, y_pred, zero_division=0),
        "Recall_Default": recall_score(y_test, y_pred, zero_division=0),
        "F1_Default": f1_score(y_test, y_pred, zero_division=0)
    })
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
results_df


Logistic Regression
Confusion Matrix:
[[10269, 5341]
 [110, 280]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.66      0.79     15610
           1       0.05      0.72      0.09       390

    accuracy                           0.66     16000
   macro avg       0.52      0.69      0.44     16000
weighted avg       0.97      0.66      0.77     16000


Random Forest
Confusion Matrix:
[[11478, 4132]
 [143, 247]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.74      0.84     15610
           1       0.06      0.63      0.10       390

    accuracy                           0.73     16000
   macro avg       0.52      0.68      0.47     16000
weighted avg       0.96      0.73      0.82     16000


XGBoost
Confusion Matrix:
[[9957, 5653]
 [89, 301]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.64     

              Model  Accuracy  ROC-AUC  Precision_Default  Recall_Default  F1_Default
            XGBoost  0.641125 0.775840           0.050554        0.771795    0.094893
      Random Forest  0.732812 0.773037           0.056406        0.633333    0.103586
Logistic Regression  0.659312 0.752736           0.049813        0.717949    0.093163

## 10. XGBoost Feature Importance

In [10]:
xgb_model = fitted_models["XGBoost"].named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": numeric_features,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

                Feature  Importance
               mat_time    0.185278
               LTV_time    0.182275
     interest_rate_time    0.101986
         FICO_orig_time    0.088869
                   time    0.072013
               gdp_time    0.069555
           balance_time    0.049346
Interest_Rate_orig_time    0.047921
               hpi_time    0.047664
      balance_orig_time    0.038742
               uer_time    0.033968
          LTV_orig_time    0.029431
          hpi_orig_time    0.028575
    REtype_PU_orig_time    0.024376
             first_time    0.000000

In [11]:
top_features = feature_importance.head(12).sort_values("Importance")
plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top XGBoost Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Observations

- The mortgage default dataset is highly imbalanced, with only about **2.44%** default records.
- Accuracy alone is not enough for this project. A model can show high accuracy by mostly predicting non-default.
- **XGBoost achieved the best ROC-AUC in this executed run**, making it a strong model for ranking default risk.
- Class imbalance was handled using `class_weight` in Logistic Regression/Random Forest and `scale_pos_weight` in XGBoost.
- The model can be improved further by threshold tuning, feature engineering, and loan-level grouped validation.

## 12. Conclusion

This notebook includes **XGBoost** along with Logistic Regression and Random Forest for mortgage default prediction. The project demonstrates data cleaning, leakage prevention, class imbalance handling, model comparison, and risk-focused evaluation.

For real-world credit-risk use, the final threshold should be chosen based on business cost because missing a true defaulter is usually more expensive than wrongly flagging a safe borrower.